# F/L/V 시간감쇠 개인 구매이력–후보상품 적합 M2 — Dunnhumby seed 42

기존 N/V 개인 구매이력–후보상품 적합 구조에 **관계 지속성(L)**만 고정 시간감쇠로 추가하는 역사적 개발구간 실험입니다.

- 학습: Dunnhumby DAY 1~683
- 평가: DAY 684~690 신규상품
- 표현: LightGCN ID 64차원 + 거래활동 적합 4차원 + 거래당 가치 적합 4차원
- 시간감쇠: $d_{uj}=\exp[-\Delta t_{uj}/\max(\overline{gap}_u,1)]$
- N 이력: 사용자 내부 상품별 장바구니 등장 비중 × 시간감쇠 후 재정규화
- V 이력: 사용자 내부 상품별 구매금액 비중 × 시간감쇠 후 재정규화
- 평균 거래간격을 계산할 수 없는 고객은 기존 N/V 이력 비중 사용
- 학습형 attention·그래프 가중·표본 가중·새 손실항 없음
- source-item과 candidate-item 표현은 분리하되 하나의 BPR로 공동학습
- 학습 시 positive 상품은 자기 이력에서 제외
- 고정: binary graph, uniform negative sampling, 100 epoch, rho=0.05
- 비교: 같은 protocol의 저장된 M1@64와 기존 N/V 이력 모형

최종 test와 holdout은 만들지 않는 seed 42 탐색이며 유의성·일반화를 주장하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'REVIEWED_SHA_PLACEHOLDER'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)


In [ ]:
import json
import torch
from lightgcn_clv_temporal_history_item_fit import (
    configure_temporal_history_item_fit_run,
    preflight_summary,
    run_temporal_history_item_fit_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_temporal_history_item_fit_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_flv_temporal_personal_history_fit_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
    current_m2_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_nv_personal_history_candidate_fit_historical_screen_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m2']['time_decay_learned'] is False
assert summary['m2']['learned_attention'] is False
assert summary['m2']['positive_item_excluded_from_training_history'] is True
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = run_temporal_history_item_fit_screen(cfg)


In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))
core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 / 기존 N/V / F/L/V 시간감쇠 핵심 비교:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)
